# _**Outputs of the WHETGEO-1D snow+soil column**_

    -Authors: GEO2GEO project (Waterjade for Rigonian Lab)
    -License: GPL v3, as WHETGEO-1D
-----
This notebook reads the outputs of the snow+soil column (`io.snowsoil`):

- the **NetCDF** file (`SnowSoilNetcdfWriter`), in the layout of the NetCDF outputs of WHETGEO-1D: dimensions `time`, `depth`, `dualDepth` (soil, $\eta$ coordinates) and `snowCell` (snow, fixed capacity, cells above the pack set to NaN);
- the **CSV** files (`SnowSoilCsvWriter`) for the comparisons with GEOtop: `point.csv` with the names and units of GEOtop's `point` file, the soil profiles `soilTz`, `psiz`, `thetaliq`, `thetaice` and the snow profiles `snowT`, `snowDepth`, `snowLiq`, `snowIce`, as GEOtop's.

Every output starts with the run header (VAL-7): code version, configuration, configuration entries and input hashes. In the NetCDF file they are global attributes, in the CSV files `#` lines.

**Conventions** (ARCH-5): $\eta$ [m] is zero at the soil surface, negative in the soil and positive in the snow; index 0 is the bottom. NetCDF in SI units; CSV in GEOtop's units (mm, °C, W/m²) with -9999 for no value.

The default input is the sample written by the unit test `SnowSoilOutputWritersTest` (`mvn -o -q test -Dtest=SnowSoilOutputWritersTest`): 20 days of the synthetic season of document 009, fidelity configuration, frozen soil with Richards.

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from WHETGEO1D_SnowSoilOutputs import *
import warnings
warnings.filterwarnings('ignore')

####### Plot settings #######
nice_fonts = {
        "legend.frameon": True,
        "legend.fancybox": True,
        "legend.facecolor": "white",
        "axes.edgecolor": "0.8",
        "axes.linewidth": 0.6,
        "grid.linewidth": 0.3,
        "text.usetex": False,
        "font.family": "serif",
        "axes.labelsize": 14,
        "font.size": 14,
        "legend.fontsize": 11,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
}
plt.rcParams.update(nice_fonts)

## Define input
**output_path**: folder with `snowsoil.nc` and the `csv` folder of the run.

In [ ]:
output_path = os.environ.get('SNOWSOIL_OUTPUT', os.path.join('..', 'target', 'snowsoil-output-sample'))
netcdf_file = os.path.join(output_path, 'snowsoil.nc')
csv_path = os.path.join(output_path, 'csv')

## Read the outputs and their header

In [ ]:
ds = read_netcdf(netcdf_file)
point, header = read_csv(os.path.join(csv_path, 'point.csv'))
for key, value in ds.attrs.items():
    print(f'{key:24s} {value}')
assert header['code_version'] == ds.attrs['code_version'], 'the two outputs come from different runs'

In [ ]:
ds

## Snow depth and snow water equivalent
The NetCDF file (SI) and the CSV point file (GEOtop units) hold the same values.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax[0].plot(ds.time, ds.snowDepth, color='C0', label='NetCDF snowDepth')
ax[0].plot(point.index.to_numpy(), point['snow_depth[mm]'].to_numpy() / 1e3, 'k:', label='CSV snow_depth[mm] / 1000')
ax[0].set_ylabel('HS [m]')
ax[0].legend()
ax[1].plot(ds.time, ds.snowWaterEquivalent, color='C1', label='NetCDF snowWaterEquivalent')
ax[1].plot(point.index.to_numpy(), point['snow_water_equivalent[mm]'].to_numpy(), 'k:', label='CSV snow_water_equivalent[mm]')
ax[1].set_ylabel('SWE [kg m$^{-2}$]')
ax[1].legend()
for a in ax:
    a.grid(color='grey', linestyle='-', linewidth=0.3)
ax[1].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.tight_layout()

## Snow profiles
Each bar is a snow cell, from its bottom to its top face in $\eta$: the cells have variable thickness and their number changes in time (`snowCells`).

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
plot_snow_cells(ds, 'snowCellTemperature', ax=ax[0], cmap='Blues_r', offset=-273.15, label='T [°C]')
plot_snow_cells(ds, 'snowCellDensity', ax=ax[1], cmap='viridis', label='density [kg m$^{-3}$]')
plot_snow_cells(ds, 'snowCellLiquidContent', ax=ax[2], cmap='magma_r', vmin=0, label='liquid content [-]')
ax[0].set_title('Snow cells')
ax[2].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.tight_layout()

In [ ]:
fig = plt.figure(figsize=(12, 3))
ds.snowCells.plot(drawstyle='steps-pre')
plt.ylabel('snow cells [-]')
plt.xlabel('')
plt.grid(color='grey', linestyle='-', linewidth=0.3)

### Profile at one date
The cells of the pack at the date with the deepest snow, drawn as steps over their thickness.

In [ ]:
k = int(ds.snowDepth.argmax())
n = int(ds.snowCells[k])
cell = ds.isel(time=k, snowCell=slice(0, n))
fig, ax = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
for a, var, scale, offset, xlabel in [(ax[0], 'snowCellTemperature', 1, -273.15, 'T [°C]'),
                                     (ax[1], 'snowCellDensity', 1, 0, 'density [kg m$^{-3}$]')]:
    v = cell[var].values * scale + offset
    y = np.r_[cell.snowCellEtaBottom.values, cell.snowCellEtaTop.values[-1]]
    a.stairs(v, y, orientation='horizontal', baseline=None, linewidth=1.8)
    a.plot(v, cell.snowCellEta, 'o', markersize=3)
    a.set_xlabel(xlabel)
    a.grid(color='grey', linestyle='-', linewidth=0.3)
ax[0].set_ylabel('$\\eta$ [m]')
fig.suptitle(str(ds.time.values[k])[:16] + f', {n} cells')

## Soil temperature and ice

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
plot_soil_profile(ds, 'T', ax=ax[0], offset=-273.15, label='T [°C]', cmap='RdBu_r', vmin=-5, vmax=5)
plot_soil_profile(ds, 'thetaIce', ax=ax[1], cmap='Blues', label='$\\theta_i$ [-]')
ax[0].set_title('Soil')
ax[1].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.tight_layout()

Define a vector of depths `myDepth` [m] (negative, $\eta$) to plot.

In [ ]:
myDepth = [-0.05, -0.15, -0.35, -0.75, -1.45]
fig = plt.figure(figsize=(12, 5))
for d in myDepth:
    (ds.T.sel(depth=d, method='nearest') - 273.15).plot(linewidth=1.8, label=f'{d} m')
plt.plot(ds.time, ds.surfaceTemperature - 273.15, 'k', linewidth=0.8, label='surface')
plt.ylabel('T [°C]')
plt.xlabel('')
plt.title('Soil temperature')
plt.legend(bbox_to_anchor=(1.02, 0.9))
plt.grid(color='grey', linestyle='-', linewidth=0.3)

The same from the CSV profile file, whose columns are the depths of the cell centres in mm, from the surface down, as GEOtop's `soilTz`.

In [ ]:
soilT, _ = read_csv(os.path.join(csv_path, 'soilTz.csv'))
depths = profile_columns(soilT)
fig = plt.figure(figsize=(12, 5))
for d in myDepth:
    column = min(depths, key=lambda c: abs(c + 1e3 * d))
    plt.plot(soilT.index.to_numpy(), soilT[column].to_numpy(), label=f'{column:.0f} mm')
plt.ylabel('T [°C]')
plt.legend(bbox_to_anchor=(1.02, 0.9))
plt.grid(color='grey', linestyle='-', linewidth=0.3)

## Surface fluxes and outflow from the pack
Fluxes are averages over the output step (NaN without snow); masses are sums over it (`cell_methods`).

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for v in ['netShortwave', 'netLongwave', 'sensibleHeat', 'latentHeat', 'surfaceHeatFlux']:
    ax[0].plot(ds.time, ds[v], linewidth=1, label=v)
ax[0].set_ylabel('[W m$^{-2}$]')
ax[0].legend(ncol=3, fontsize=10)
for v in ['melt', 'snowRunoff', 'soilInput', 'infiltration']:
    ax[1].plot(ds.time, ds[v], linewidth=1, label=v)
ax[1].set_ylabel('[kg m$^{-2}$ per step]')
ax[1].legend(ncol=4, fontsize=10)
for a in ax:
    a.grid(color='grey', linestyle='-', linewidth=0.3)
ax[1].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.tight_layout()

## Diagnostics (VAL-7)
The closures of the snowpack operators, of the heat solve and of the soil, and the numerical corrections, each on its own, cumulated over the run.

In [ ]:
diag = ['snowMassClosure', 'snowEnergyClosure', 'heatClosure', 'linearisationEnergy', 'soilWaterBalanceError',
        'soilEnergyClosure', 'soilStorageHeat']
pd.set_option('display.float_format', '{:.3e}'.format)
summary = pd.DataFrame({v: {'units': ds[v].attrs['units'], 'max |step|': float(np.nanmax(np.abs(ds[v]))),
                            'cumulated': float(np.nansum(ds[v]))} for v in diag}).T
print(f"steps: {int(ds.steps.sum())}, heat not converged: {int(ds.heatNotConverged.sum())}, "
      f"surface not converged: {int(ds.surfaceNotConverged.sum())}")
summary

## Alignment with GEOtop
The columns of `point.csv` that GEOtop's `point` file also has keep GEOtop's name and units. Set `geotop_point` to a GEOtop point file (for instance the Matsch_P2_Ref_007 reference, `casi/Matsch_P2_Ref_007/README.md`) to list them; the comparison itself needs a run of the column on the same case.

In [ ]:
geotop_point = os.environ.get('GEOTOP_POINT', '')
if os.path.exists(geotop_point):
    gt, _ = read_csv(geotop_point)
    common = [c for c in point.columns if c in gt.columns and c != DATE_COLUMN]
    print(len(common), 'common columns:', ', '.join(common))
else:
    print('GEOtop columns of point.csv:', ', '.join(c for c in point.columns if not c[0].islower() or '_' in c))